# Optimized LLM Fine Tuning Experiment

My sufficient statistic regularizer (SSR) framework and associated theory provides optimal statistical efficiency guarantees.
The applied impact of improved statistical efficiency must be observed; it can't be derived. 
In this experiment, I try to see if something new is possible: LLM tuning with very small datasets. 
For example, could an agent learn from a single user? 
Can information be retained in model parameters so retrieval isn't limited by context windows? 
We have to try and see. 

### Learning `torch.distributed`

In [4]:
%pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 33.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 86.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 48.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 87.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 90.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 88.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 98.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 97.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch.distributed.fsdp as fsdp

In [6]:
import sys
import os
import shutil 

python_executable = sys.executable
torchrun_executable = shutil.which("torchrun", path=f"{os.path.dirname(python_executable)}")
print(f'torchrun executable: {torchrun_executable}')

torchrun executable: /anaconda/envs/azureml_py310_sdkv2/bin/torchrun


In [1]:
import subprocess
import sys
import os
import shutil 

def run_torch_script(script_path, nproc, *args): 
    """Finds the correct torchrun and runs the script."""
    python_executable = sys.executable
    torchrun_executable = shutil.which("torchrun", path=f"{os.path.dirname(python_executable)}") 

    command = [torchrun_executable, "--nnodes=1", f"--nproc_per_node={nproc}", script_path] + list(map(str, args)) 

    print("Running: " + " ".join(command))
    
    result = subprocess.run(command, text=True, capture_output=True)
    
    print("\n--- Script Output ---")
    if result.stdout:
        print(result.stdout)
    
    if result.stderr:
        print("\n--- Script Errors ---")
        print(result.stderr)

    return result.returncode

In [2]:
# Run it:
os.environ["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL" ## Debugging 
run_torch_script("torch_dist_example.py", nproc=2)

Running: /anaconda/envs/azureml_py310_sdkv2/bin/torchrun --nnodes=1 --nproc_per_node=2 torch_dist_example.py


KeyboardInterrupt: 